# Lab 2: Working with the Anthropic API and Reasoning Models

## Overview

This notebook explores the Anthropic API, covering both standard chat capabilities and Claude's extended thinking (reasoning) features. It shows how to configure the client, steer behavior with system prompts, and use a reasoning model to evaluate the output of a standard model.

The example use case is **customer service**: a standard model drafts a reply to a customer complaint based on company policy, and a reasoning model then evaluates that reply for appropriateness, tone, and policy alignment.

## Objectives

- Set up and configure the Anthropic API
- Create tailored system prompts for specific use cases
- Compare standard responses with reasoning-enhanced outputs
- Inspect and use the extended thinking process in reasoning models

## Prerequisites

- An Anthropic API key set as the environment variable `ANTHROPIC_API_KEY` (a `.env` file works well)
- The `anthropic`, `python-dotenv`, and `ipython` packages installed

## Step 1: Import Libraries and Load Environment Variables

In [1]:
import os
import anthropic
from dotenv import load_dotenv
from IPython.display import Markdown, display

load_dotenv()

ANTHROPIC_API_KEY = os.getenv("ANTHROPIC_API_KEY")

if ANTHROPIC_API_KEY is None:
    raise Exception("API key is missing. Set ANTHROPIC_API_KEY in your environment or a .env file.")

## Step 2: Make Your First API Call

Initialize the client and send a basic message. The `system` prompt steers the model's behavior — here we deliberately instruct it to be unhelpful to demonstrate how strongly the system prompt shapes the response.

In [2]:
client = anthropic.Client()

message = client.messages.create(
    model="claude-3-5-haiku-latest",
    system="You are an unhelpful assistant.",
    messages=[
        {"role": "user", "content": "What is the capital of France?"}
    ],
    max_tokens=100
)

print(message.content[0].text)

I do not want to be helpful, so I will not tell you the capital of France.


## Step 3: Customize the Prompt

Build a customer service use case. A standard model (`claude-3-5-haiku-latest`) drafts a reply to a customer complaint, guided by company policy and tone requirements supplied in the prompt.

In [3]:
complaint = "You have overcharged me for this product. I demand a refund."

In [4]:
company_policy = """
- We offer refunds only within 14 days of purchase.
- For purchases over $100, a supervisor must approve the refund.
- We prioritize customer satisfaction but aim to minimize refund abuse.
"""

In [5]:
system_message = "You are a customer service agent."

prompt = f"""
We have a user complaint: "{complaint}"
Company policy is as follows: {company_policy}

Your job is to respond to the user complaint based on the company policy.
Keep your tone professional and empathetic, but also firm about the policy.
Keep your response short and to the point, no more than 50 words.
"""

chat_response = client.messages.create(
    model="claude-3-5-haiku-latest",
    system=system_message,
    messages=[
        {"role": "user", "content": prompt}
    ],
    max_tokens=200
)

chat_message = chat_response.content[0].text
print(chat_message)

I apologize for your frustration. To proceed with a refund, could you please provide your purchase date and order number? We'll review your case carefully and ensure a fair resolution in line with our policy.


## Step 4: Evaluate the Response Using a Reasoning Model

Use a reasoning model (`claude-3-7-sonnet-latest`) with **extended thinking** enabled to evaluate the draft reply against the complaint and company policy. With thinking enabled, the response contains two blocks: a `ThinkingBlock` (the model's internal reasoning) and a `TextBlock` (the final answer).

In [8]:
system_message = "You are a senior customer service agent with advanced reasoning capabilities. You analyze customer complaints and evaluate responses based on company policy."

reasoning_prompt = f"""
A customer has made a complaint:
{complaint}

Evaluate the following customer service response:
{chat_message}

Company Policy:
{company_policy}

Provide an analysis that covers:
- Whether the response is appropriate given the complaint and company policy
- The professional tone and empathy of the response
"""

reasoning_response = client.messages.create(
    model="claude-3-7-sonnet-latest",
    thinking={
        "type": "enabled",
        "budget_tokens": 2000
    },
    max_tokens=3000,
    temperature=1,
    system=system_message,
    messages=[
        {"role": "user", "content": reasoning_prompt}
    ]
)

# Inspect the raw response: a ThinkingBlock followed by a TextBlock
print(reasoning_response.content)

[ThinkingBlock(signature='ErUBCkYIBhgCIkAegwjkTkgzzdVwgJw6u/3c0h+TEQTOWH5EqGYH30LhAmCaFjl1GMxoe/99+Yltk1kfpQ2ovePPROLDhMwxoKb4EgxIxoZ/adgHmuE3Gs8aDP0ldWYlEOCEfPRbFCIwndHHhs2+lS846JMaoHsVjbhJEhpERuTvuYS7TxLOwu1IOPOveIWJA5i+iOHRp/ilKh31DVUXDaZ9hj6LvoMZw4f4criyr/WV9q1vkybDuxgC', thinking="Let me analyze this customer service situation:\n\n1. The customer complaint:\n   - Very brief complaint stating they were overcharged\n   - Demanding a refund\n   - No specific details provided (no order number, purchase date, product information, or amount)\n\n2. The email response being evaluated:\n   - Acknowledges the customer's frustration\n   - Requests necessary information (purchase date and order number)\n   - Promises to review the case and find a fair resolution\n   - Mentions alignment with company policy\n\n3. Company policy details:\n   - 14-day refund window\n   - Supervisor approval needed for purchases over $100\n   - Balance between customer satisfaction and preventing refund abuse\n\n

In [11]:
# Extract the text block (the model's final answer) and render it as Markdown
reasoning_text = next(block.text for block in reasoning_response.content if block.type == "text")

display(Markdown(reasoning_text))

# Analysis of Customer Service Response

## Appropriateness Given Complaint and Company Policy

The response is **appropriate and well-balanced** for this situation because:

- It correctly avoids immediately promising a refund since the customer provided no details to verify eligibility against the 14-day policy
- It requests specific information (purchase date and order number) necessary to determine if the refund request falls within company guidelines
- It mentions policy alignment, setting appropriate expectations without being rigid
- It neither denies nor guarantees a refund before having all relevant information, which protects the company while keeping resolution options open

## Professional Tone and Empathy

The response demonstrates strong customer service qualities:

- Opens with an acknowledgment of the customer's frustration, showing empathy toward their concern
- Maintains a professional, non-defensive tone despite the demanding nature of the complaint
- Provides clear next steps rather than leaving the interaction open-ended
- Strikes a good balance between being helpful and maintaining company procedures
- Communicates briefly but effectively without unnecessary explanations

## Overall Assessment

This is an exemplary initial response that handles a vague complaint professionally. It shows empathy while gathering necessary information to evaluate the claim according to policy. The agent has successfully balanced customer service with proper procedure.

## Summary

In this lab we configured the Anthropic client, observed how the system prompt steers model behavior, and built a customer service workflow. A standard model (`claude-3-5-haiku-latest`) drafted a policy-aware reply to a complaint, and a reasoning model (`claude-3-7-sonnet-latest`) with extended thinking enabled evaluated that reply.

Key takeaway: extended thinking returns both a reasoning trace and a final answer as separate content blocks, so the text block must be extracted explicitly. Reasoning models are well-suited to evaluation and analysis tasks that benefit from explicit, step-by-step deliberation, while standard chat models are faster and ideal for direct response generation.